# Audio Feature Extraction — MELD

Encoder: configurable via `MODEL_NAME` in the config cell (default: `microsoft/wavlm-large`)  
Feature: masked mean-pool over last hidden state frames → shape `(HIDDEN_SIZE,)` per utterance  
Output: `Dataset/Processed/MELD/features/audio_{MODEL_TAG}_{split}.pt`  
Format: `dict {clip_name: np.array(HIDDEN_SIZE,)}` per split  

Expected counts: train≈9988, dev≈1108, test=2610  
Note: `dia125_utt3` (train) audio permanently missing — skipped automatically.

In [1]:
import torch
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoFeatureExtractor, AutoModel
from tqdm.notebook import tqdm

In [2]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/MELD"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 1      # keep low — variable-length audio padded per batch, 12 GB VRAM
SAMPLE_RATE = 16000  # both WavLM-Large and HuBERT-Large expect 16 kHz mono
SPLITS      = ["train", "dev", "test"]

# ── Swap model here ───────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/wavlm-large"      # any HuggingFace AutoModel-compatible SSL audio encoder
# Examples:
#   "microsoft/wavlm-large"           → audio_microsoft_wavlm_large_{split}.pt       (hidden=1024)
#   "facebook/hubert-large-ll60k"     → audio_facebook_hubert_large_ll60k_{split}.pt  (hidden=1024)
#   "facebook/hubert-base-ls960"      → audio_facebook_hubert_base_ls960_{split}.pt   (hidden=768)
#   "facebook/wav2vec2-large-960h"    → audio_facebook_wav2vec2_large_960h_{split}.pt (hidden=1024)
# ─────────────────────────────────────────────────────────────────────────────
MODEL_TAG = MODEL_NAME.replace("/", "_").replace("-", "_")
print(f"Model    : {MODEL_NAME}")
print(f"Tag      : {MODEL_TAG}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Model    : microsoft/wavlm-large
Tag      : microsoft_wavlm_large
Device   : cuda
GPU      : NVIDIA GeForce RTX 3060
VRAM free: 11.9 GB


In [3]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances : {len(labels)}")
print(f"Columns          : {labels.columns.tolist()}")
print(f"\nSplit counts:")
print(labels["split"].value_counts())
labels.head(3)

Total utterances : 13708
Columns          : ['clip_name', 'split', 'dia_id', 'utt_id', 'utterance', 'speaker', 'emotion', 'sentiment', 'season', 'episode', 'start_time', 'end_time', 'audio_path', 'video_path', 'text_path', 'status']

Split counts:
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64


,clip_name,split,dia_id,utt_id,utterance,speaker,emotion,sentiment,season,episode,start_time,end_time,audio_path,video_path,text_path,status
0,dia0_utt0,train,0,0,also I was the point person on my companys tr...,Chandler,neutral,neutral,8,21,"00:16:16,059","00:16:21,731",audio/train/dia0_utt0.wav,video/train/dia0_utt0.mp4,text/train/dia0_utt0.txt,ok
1,dia0_utt1,train,0,1,You mustve had your hands full.,The Interviewer,neutral,neutral,8,21,"00:16:21,940","00:16:23,442",audio/train/dia0_utt1.wav,video/train/dia0_utt1.mp4,text/train/dia0_utt1.txt,ok
2,dia0_utt2,train,0,2,That I did. That I did.,Chandler,neutral,neutral,8,21,"00:16:23,442","00:16:26,389",audio/train/dia0_utt2.wav,video/train/dia0_utt2.mp4,text/train/dia0_utt2.txt,ok


In [4]:
processor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
model = model.to(DEVICE)
HIDDEN_SIZE = model.config.hidden_size
print(f"Hidden size : {HIDDEN_SIZE}")
print(f"Params      : {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Hidden size : 1024
Params      : 315M


In [5]:
import math

CHUNK_S = 30  # max chunk duration fed to model in one forward pass

def load_wav(path: Path) -> np.ndarray:
    """Load wav, resample to SAMPLE_RATE if needed, return mono float32."""
    wav, sr = torchaudio.load(str(path))
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    return wav.mean(0).numpy()  # (T,)


def _model_forward(waveform: np.ndarray) -> np.ndarray:
    """Single forward pass on one waveform segment → (HIDDEN_SIZE,)."""
    inputs = processor(
        [waveform],
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state  # (1, T, H)
    if "attention_mask" in inputs:
        feat_len = model._get_feat_extract_output_lengths(
            inputs["attention_mask"].sum(-1)
        ).long()
        T = hidden.shape[1]
        frame_mask = (
            torch.arange(T, device=DEVICE).unsqueeze(0) < feat_len.unsqueeze(1)
        ).unsqueeze(-1).float()  # (1, T, 1)
        feat = (hidden * frame_mask).sum(1) / frame_mask.sum(1)  # (1, H)
    else:
        feat = hidden.mean(1)
    return feat.cpu().numpy()[0]  # (H,)


def embed_waveform(wav: np.ndarray, name: str = "") -> np.ndarray:
    """Chunk wav into CHUNK_S segments, embed each, return mean → (HIDDEN_SIZE,)."""
    chunk_samples = CHUNK_S * SAMPLE_RATE
    n_chunks = math.ceil(len(wav) / chunk_samples)
    if n_chunks > 1:
        print(f"  [info] {name}: {len(wav)/SAMPLE_RATE:.1f}s → {n_chunks} chunks")
    chunks = [wav[i * chunk_samples:(i + 1) * chunk_samples] for i in range(n_chunks)]
    feats = np.stack([_model_forward(c) for c in chunks])  # (n_chunks, H)
    return feats.mean(0)  # (H,)


def extract_batch(audio_paths: list[Path]) -> np.ndarray:
    """Returns features shape (B, HIDDEN_SIZE). Handles long files via chunking."""
    return np.stack([
        embed_waveform(load_wav(p), name=p.name)
        for p in audio_paths
    ])

In [ ]:
all_skipped: dict[str, list[str]] = {}

for split in SPLITS:
    split_df    = labels[labels["split"] == split].reset_index(drop=True)
    clip_names  = split_df["clip_name"].tolist()
    audio_paths = split_df["audio_path"].tolist()
    print(f"\n--- {split} ({len(split_df)} utterances) ---")

    features: dict[str, np.ndarray] = {}
    skipped:  list[str] = []

    batch_ids:   list[str]  = []
    batch_paths: list[Path] = []

    def flush_batch():
        if not batch_paths:
            return
        feats = extract_batch(batch_paths)
        for bid, feat in zip(batch_ids, feats):
            features[bid] = feat
        batch_ids.clear()
        batch_paths.clear()

    for cname, apath in tqdm(zip(clip_names, audio_paths), total=len(clip_names), desc=split):
        p = DATA_ROOT / apath  # paths in labels.csv are relative to DATA_ROOT
        if not p.exists():
            skipped.append(cname)
            continue
        batch_ids.append(cname)
        batch_paths.append(p)
        if len(batch_paths) == BATCH_SIZE:
            flush_batch()

    flush_batch()  # remainder

    out_path = FEATURES_DIR / f"audio_{MODEL_TAG}_{split}.pt"
    torch.save(features, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"Saved    : {out_path}  ({size_mb:.1f} MB, {len(features)} entries)")
    if skipped:
        print(f"Skipped  : {skipped}")
    all_skipped[split] = skipped


--- test (2610 utterances) ---


test:   0%|          | 0/2610 [00:00<?, ?it/s]

/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  [info] dia38_utt4.wav: 305.0s → 11 chunks
  [info] dia220_utt0.wav: 235.1s → 8 chunks
Saved    : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/features/audio_microsoft_wavlm_large_test.pt  (16.3 MB, 2610 entries)


In [ ]:
print("=== Verification ===")

for split in SPLITS:
    out_path   = FEATURES_DIR / f"audio_{MODEL_TAG}_{split}.pt"
    loaded     = torch.load(out_path, weights_only=False)
    split_df   = labels[labels["split"] == split]
    n_skipped  = len(all_skipped.get(split, []))
    expected   = len(split_df) - n_skipped

    assert len(loaded) == expected, f"{split}: Expected {expected}, got {len(loaded)}"

    sample_key  = next(iter(loaded))
    sample_feat = loaded[sample_key]
    assert sample_feat.shape == (HIDDEN_SIZE,), f"{split}: shape {sample_feat.shape}"

    all_feats = np.stack(list(loaded.values()))
    has_nan   = np.isnan(all_feats).any()
    has_inf   = np.isinf(all_feats).any()
    assert not has_nan, f"{split}: NaN detected"
    assert not has_inf, f"{split}: Inf detected"

    l2_norm  = np.linalg.norm(sample_feat)
    split_row = labels[labels.clip_name == sample_key].iloc[0]
    print(f"\n[{split}]")
    print(f"  Count   : {len(loaded)} / {len(split_df)} ({n_skipped} skipped)  ✓")
    print(f"  Shape   : {sample_feat.shape}  ✓")
    print(f"  Mean    : {all_feats.mean():.4f}")
    print(f"  Std     : {all_feats.std():.4f}")
    print(f"  Has NaN : {has_nan}  ✓")
    print(f"  Has Inf : {has_inf}  ✓")
    print(f"  Sample  : {sample_key}  |  emotion={split_row.emotion}  |  L2={l2_norm:.2f}")

print("\nAll assertions passed.")

=== Verification ===

[test]
  Count   : 2610 / 2610 (0 skipped)  ✓
  Shape   : (1024,)  ✓
  Mean    : -0.0004
  Std     : 0.0858
  Has NaN : False  ✓
  Has Inf : False  ✓
  Sample  : dia0_utt0  |  emotion=neutral  |  L2=1.90

All assertions passed.
